In [ ]:
import torch
import torchvision
import numpy as np
import cv2
import matplotlib.pyplot as plt

In [ ]:
# select device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
# upload the exported model
from google.colab import files
print('upload watch_detector_yolo3u_traced.pth')
_ = files.upload()

In [ ]:
# or download it
#!wget http://www.agentspace.org/download/watch_detector_yolo3u_traced.pth

In [ ]:
# load model
model = torch.jit.load('watch_detector_yolo3u_traced.pth')

In [ ]:
model

In [ ]:
import cv2
import numpy as np

In [ ]:
# -------------- preprocessing------------------------

In [ ]:
def preprocess(img, img_size=416, border_color=114):
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # convert to RGB

    # Resize with unchanged aspect ratio (letterbox)
    h, w = img.shape[:2]
    scale = min(img_size / h, img_size / w)
    nh, nw = int(h * scale), int(w * scale)
    img_resized = cv2.resize(img, (nw, nh))
    new_img = np.full((img_size, img_size, 3), border_color, dtype=np.uint8)
    new_img[:nh, :nw] = img_resized

    new_img = new_img.astype(np.float32) / 255.0 # normalize into 0-1
    new_img = np.transpose(new_img, (2, 0, 1)) # convert H,W,C to C,H,W
    tensor = torch.from_numpy(new_img).unsqueeze(0) # turn to tensor B,C,H,W
    return tensor, scale, nw, nh

In [ ]:
# -------------- postprocessing------------------------

In [ ]:
def xywh2xyxy(x):
    y = x.clone()
    y[..., 0] = x[..., 0] - x[..., 2] / 2  # x1
    y[..., 1] = x[..., 1] - x[..., 3] / 2  # y1
    y[..., 2] = x[..., 0] + x[..., 2] / 2  # x2
    y[..., 3] = x[..., 1] + x[..., 3] / 2  # y2
    return y

In [ ]:
# Non-Maximum Suppression
import torchvision

def non_max_suppression (prediction, conf_thres = 0.25, iou_thres = 0.45, max_det = 300, max_wh = 7680):
    bs = prediction.shape[0]  # batch size (BCN, i.e. 1,84,6300)
    nc = prediction.shape[1] - 4  # number of classes
    extra = prediction.shape[1] - nc - 4  # number of extra info
    mi = 4 + nc  # mask start index
    xc = prediction[:, 4:mi].amax(1) > conf_thres  # candidates
    xinds = torch.arange(prediction.shape[-1], device=prediction.device).expand(bs, -1)[..., None]  # to track idxs

    prediction = prediction.transpose(-1, -2)  # shape(1,84,6300) to shape(1,6300,84)
    prediction[..., :4] = xywh2xyxy(prediction[..., :4])  # xywh to xyxy

    output = [torch.zeros((0, 6 + extra), device=prediction.device)] * bs
    keepi = [torch.zeros((0, 1), device=prediction.device)] * bs  # to store the kept idxs
    for xi, (x, xk) in enumerate(zip(prediction, xinds)):  # image index, (preds, preds indices)
        # Apply constraints
        filt = xc[xi]  # confidence
        x = x[filt]

        # If none remain process next image
        if not x.shape[0]:
            continue

        # Detections matrix nx6 (xyxy, conf, cls)
        box, cls, mask = x.split((4, nc, extra), 1)

        conf, j = cls.max(1, keepdim=True) # best class only
        filt = conf.view(-1) > conf_thres
        x = torch.cat((box, conf, j.float(), mask), 1)[filt]

        c = x[:, 5:6] * max_wh  # classes
        scores = x[:, 4]  # scores
        boxes = x[:, :4] + c  # boxes (offset by class)

        i = torchvision.ops.nms(boxes, scores, iou_thres) # calling nms

        i = i[:max_det]  # limit detections
        output[xi] = x[i]

    return output

In [ ]:
def postprocess(predictions, orig, scale, nw, nh, conf_thres=0.75, iou_thres=0.45):
    pred = non_max_suppression(predictions, conf_thres=conf_thres, iou_thres=iou_thres)[0]
    oh, ow = orig[:2]
    pred[:, [0,2]] = pred[:, [0,2]] / nw * ow
    pred[:, [1,3]] = pred[:, [1,3]] / nh * oh
    return pred

In [ ]:
# -------------- inference ------------------------

In [ ]:
# upload image
from google.colab import files
print('upload image')
uploaded = files.upload()
imagefile = list(uploaded.keys())[0]
print(imagefile)

In [ ]:
#!wget http://www.agentspace.org/download/watch.jpg
#imagefile = 'watch.jpg'

In [ ]:
img = cv2.imread(imagefile)

In [ ]:
from google.colab.patches import cv2_imshow
cv2_imshow(img)

In [ ]:
x, scale, nw, nh = preprocess(img)

In [ ]:
x = x.to("cuda")
y = model(x)
y = y[0].cpu()

In [ ]:
pred = postprocess(y, img.shape, scale, nw, nh, conf_thres=0.75, iou_thres=0.45)

In [ ]:
pred

In [ ]:
def plot_one_box(xyxy, img, label=None, color=(0,255,0), line_thickness=None): # Plots one bounding box on the image
    # Calculate line thickness based on image size
    tl = line_thickness or round(0.002 * (img.shape[0] + img.shape[1]) / 2) + 1
    # Convert bounding box coordinates to integers
    pt1, pt2 = (int(xyxy[0]), int(xyxy[1])), (int(xyxy[2]), int(xyxy[3]))
    # Draw the bounding box rectangle on the image
    cv2.rectangle(img, pt1, pt2, color, thickness=tl)
    if label:
        # Calculate font thickness
        tf = max(tl - 1, 1)
        # Calculate text size
        t_size = cv2.getTextSize(label, 0, fontScale=tl / 3, thickness=tf)[0]
        # Calculate the coordinates for the label background rectangle
        pt3 = pt1[0] + t_size[0], pt1[1] - t_size[1] - 3
        # Draw the label background rectangle
        cv2.rectangle(img, pt1, pt3, color, cv2.FILLED)
        # Draw the label text
        cv2.putText(img, label, (pt1[0], pt1[1] - 2), 0, tl / 3, (0,0,0), thickness=tf)

In [ ]:
names=['watch']

In [ ]:
disp = np.copy(img)
for detection in pred:
    *xyxy, confidence, classid  = detection
    print(f'{confidence.item():.2f},{int(xyxy[0].item())},{int(xyxy[1].item())},{int(xyxy[2].item())},{int(xyxy[3].item())},{names[classid.int().item()]}')
    plot_one_box(xyxy, disp, label=names[classid.int().item()], color=(0,255,0))

In [ ]:
#display result
from google.colab.patches import cv2_imshow
cv2_imshow(disp)

In [ ]:
#all in one
def process_image(img, conf_thresh=0.75, iou_thresh=0.45):
    x, scale, nw, nh = preprocess(img)
    x = x.to(device)
    y = model(x)
    y = y[0].cpu()
    pred = postprocess(y, img.shape, scale, nw, nh, conf_thresh, iou_thresh)
    disp = np.copy(img)
    for detection in pred:
        *xyxy, confidence, classid  = detection
        plot_one_box(xyxy, disp, label=names[classid.int().item()], color=(0,255,0))
    return disp

In [ ]:
# upload video
#!wget http://www.agentspace.org/download/watch.avi
#videofile = 'watch.avi'

In [ ]:
# upload video
from google.colab import files
print('upload video')
uploaded = files.upload()
videofile = list(uploaded.keys())[0]
print(videofile)

In [ ]:
# process video
resultfile = 'result.avi'
video = cv2.VideoCapture(videofile)
fps = video.get(cv2.CAP_PROP_FPS)
hasFrame, frame = video.read()
out = cv2.VideoWriter()
out.open(resultfile,cv2.VideoWriter_fourcc('M','J','P','G'),fps,(frame.shape[1],frame.shape[0]))
while True:
    result = process_image(frame, conf_thresh=0.96, iou_thresh=0.45)
    out.write(result)
    hasFrame, frame = video.read()
    if not hasFrame:
        break
out.release()

In [ ]:
# download video
files.download(resultfile)